In [1]:
import os
import glob
import pickle
import cv2
import numpy as np

# Set hyperparameters and define relative paths for input/output
ORB_FEATURES = 500
GALLERY_FOLDER = "../data/raw" 
MAP_FILE = "../build/image_map.pkl"

# Initialize the ORB algorithm to extract a maximum of 500 features per image
orb = cv2.ORB_create(nfeatures=ORB_FEATURES)

# Initialize the data structures that will form our global database
all_descriptors = []
image_id_map = []
image_filenames = {}

# Locate all valid image files in the gallery folder
image_paths = glob.glob(os.path.join(GALLERY_FOLDER, "*.*"))
valid_extensions = ('.png', '.jpg', '.jpeg')
image_paths = [p for p in image_paths if p.lower().endswith(valid_extensions)]

print(f"--- Found {len(image_paths)} images in '{GALLERY_FOLDER}'. Starting feature extraction ---")

for img_id, filepath in enumerate(image_paths):
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Warning: Could not read {filepath}. Skipping.")
        continue

    # Extract the mathematical "fingerprints" (descriptors) from the current image
    _, descriptors = orb.detectAndCompute(img, None)

    # If features were successfully found, add them to our global database structures
    if descriptors is not None:
        all_descriptors.append(descriptors)
        
        # Create a tracker array linking every single extracted feature back to this specific img_id
        image_id_map.extend([img_id] * len(descriptors))
        
        # Store the human-readable filename for later use
        image_filenames[img_id] = os.path.basename(filepath)
        
        print(f"Indexed {os.path.basename(filepath)} (ID: {img_id}) with {len(descriptors)} features.")

if not all_descriptors:
    print("Error: No images found in data/raw! Did you upload them?")
else:
    # Stack all individual image feature lists into one massive 2D matrix
    global_descriptor_matrix = np.vstack(all_descriptors)
    image_id_map = np.array(image_id_map, dtype=np.int32)
    
    print(f"\nTotal global descriptors: {global_descriptor_matrix.shape[0]}")
    print("Saving descriptor database...")
    
    # Serialize (freeze) the data structures and save them to the hard drive as a binary file
    with open(MAP_FILE, "wb") as f:
        pickle.dump({
            "image_id_map": image_id_map, 
            "image_filenames": image_filenames,
            "global_descriptor_matrix": global_descriptor_matrix
        }, f)
    
    print("\nSUCCESS! Your database files are securely saved inside the 'build' folder.")

--- Found 3000 images in '../data/raw'. Starting feature extraction ---
Indexed azuki_#784.png (ID: 0) with 500 features.
Indexed bayc_#1393.png (ID: 1) with 500 features.
Indexed bayc_#1890.png (ID: 2) with 500 features.
Indexed azuki_#711.png (ID: 3) with 500 features.
Indexed azuki_#934.png (ID: 4) with 500 features.
Indexed bayc_#1406.png (ID: 5) with 500 features.
Indexed azuki_#263.png (ID: 7) with 500 features.
Indexed azuki_#89.png (ID: 8) with 500 features.
Indexed bayc_#1507.png (ID: 10) with 500 features.
Indexed bayc_#1811.png (ID: 12) with 500 features.
Indexed azuki_#977.png (ID: 13) with 500 features.
Indexed azuki_#959.png (ID: 14) with 500 features.
Indexed azuki_#570.png (ID: 16) with 500 features.
Indexed bayc_#1702.png (ID: 17) with 500 features.
Indexed azuki_#926.png (ID: 19) with 500 features.
Indexed azuki_#992.png (ID: 22) with 500 features.
Indexed bayc_#1282.png (ID: 24) with 500 features.
Indexed bayc_#1284.png (ID: 26) with 500 features.
Indexed bayc_#1063.